# Stage 6 & 7: Algorithm 2 — Random Forest Classifier

**Owner:** Member B  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/random_forest_tuned.pkl`  

## Scope
Random Forest is a **bagging ensemble** of decision trees. Each tree is trained on a random bootstrap sample
of the training data, and predictions are averaged across all trees, reducing variance and overfitting.

Key advantages over single Decision Trees:
- Reduces overfitting through bagging (bootstrap aggregating)
- Handles non-linear feature interactions (e.g. distance × item count)
- `class_weight='balanced_subsample'` addresses class imbalance at each bootstrap sample

This notebook:
1. Loads the final modeling datasets from `data/final/`
2. Trains a single Decision Tree baseline
3. Trains a default Random Forest baseline
4. Performs 5-Fold Stratified RandomizedSearchCV tuning on `X_train` only
5. Evaluates the tuned model on `X_val` with full metrics
6. Reports feature importances and exports the tuned model artifact


## 1. Imports & Setup

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)

FINAL_DIR  = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train : {X_train.shape} | Late rate: {y_train.mean()*100:.2f}%')
print(f'X_val   : {X_val.shape}   | Late rate: {y_val.mean()*100:.2f}%')


X_train : (67529, 67) | Late rate: 9.03%
X_val   : (14470, 67)   | Late rate: 5.34%


## 2. Decision Tree Baseline
A single shallow Decision Tree establishes the baseline for tree-based models.

In [6]:
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
dt_probs = dt.predict_proba(X_val)[:, 1]

print('=== DECISION TREE BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, dt_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, dt_probs):.4f}')


=== DECISION TREE BASELINE ===
Val ROC-AUC : 0.6459
Val PR-AUC  : 0.0855


## 3. Default Random Forest Baseline
An untuned Random Forest baseline measures the benefit of hyperparameter search.

In [7]:
rf_default = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_default.fit(X_train, y_train)
default_probs = rf_default.predict_proba(X_val)[:, 1]

print('=== DEFAULT RANDOM FOREST BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, default_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, default_probs):.4f}')


=== DEFAULT RANDOM FOREST BASELINE ===
Val ROC-AUC : 0.7076
Val PR-AUC  : 0.1254


## 4. Hyperparameter Tuning — 5-Fold Stratified RandomizedSearchCV
**Parameters tuned:**
- `n_estimators`: Number of trees. More trees = lower variance but slower training.
- `max_depth`: Maximum tree depth. Controls overfitting.
- `min_samples_split`: Minimum samples to split a node. Higher = simpler trees.
- `max_features`: Features considered at each split (`sqrt` = standard best practice).
- `class_weight`: `balanced_subsample` recalculates weights per bootstrap sample.

**Note:** RandomizedSearchCV is used instead of GridSearchCV because the search space is large.
It samples 10 random parameter combinations rather than exhaustively testing all combinations.
Training time for Random Forest is longer due to building 300 trees on 67,000+ rows.
This is expected and normal behaviour for ensemble models.

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=10, scoring='average_precision',
    cv=cv, random_state=42, n_jobs=-1, verbose=1
)
rand.fit(X_train, y_train)

print(f'Best Parameters : {rand.best_params_}')
print(f'Best CV PR-AUC  : {rand.best_score_:.4f}')
best_rf = rand.best_estimator_


Fitting 5 folds for each of 10 candidates, totalling 50 fits


## 5. Validation Set Evaluation

In [ ]:
val_probs = best_rf.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print('=== TUNED RANDOM FOREST — VALIDATION SET METRICS ===')
print(f'Val ROC-AUC        : {roc_auc_score(y_val, val_probs):.4f}')
print(f'Val PR-AUC         : {average_precision_score(y_val, val_probs):.4f}')
print(f'Val F1-Score (0.5) : {f1_score(y_val, val_preds):.4f}')
print(f'Val Recall   (0.5) : {recall_score(y_val, val_preds):.4f}')
print(f'Val Precision(0.5) : {precision_score(y_val, val_preds):.4f}')

print('\nClassification Report:')
print(classification_report(y_val, val_preds, target_names=['On-Time (0)', 'Late (1)']))

cm = confusion_matrix(y_val, val_preds)
print(f'Confusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')


## 6. Feature Importance (Gini Impurity)
Random Forest measures feature importance as the mean decrease in Gini Impurity across all trees.
Features with higher importance contribute more to reducing prediction uncertainty.

In [ ]:
feat_imp = pd.Series(best_rf.feature_importances_, index=X_train.columns)
top10 = feat_imp.sort_values(ascending=False).head(10)

print('Top 10 Most Important Features (Gini Impurity Decrease):')
for feat, imp in top10.items():
    print(f'  {feat:<45} : {imp:.4f}')


## 7. Save Tuned Model Artifact

In [ ]:
joblib.dump(best_rf, MODELS_DIR / 'random_forest_tuned.pkl')
print('Saved: models/random_forest_tuned.pkl')
print(f'n_estimators : {best_rf.n_estimators}')
print(f'max_depth    : {best_rf.max_depth}')
print(f'class_weight : {best_rf.class_weight}')
